# Ubuntu-Aligned Latent Probing for Toxicity Detection with Gemma 2B
#
# This notebook:
# 1. Loads 4,000 English-Amharic Ubuntu-oriented synthetic dataset.
# 2. Creates paired English + Amharic examples without using the rationale/harm
# 3. Loads Gemma 2 2B.
# 4. Extracts mean-pooled hidden states from every transformer layer.
# 5. Trains a linear probe (StandardScaler + LogisticRegression) at each layer.
# 6. Selects the best layer using a GROUPED validation split, so duplicate
# 7. Saves the selected probe as trained_probe.joblib for classifier.py.

In [1]:
%%capture
!pip install jax[cuda12]==0.7.2
# Packages used.
import os # Used for setting Keras configuration variables.
os.environ["KERAS_BACKEND"] = "jax" # Set a parameter for Keras.
import re # Used for splitting text on whitespace.

import keras # Used for defining an training the model.
import pandas as pd # Used for loading the dataset.
import tensorflow as tf # Used for shuffling the dataset.

# The following line provides configuration for Keras.
keras.utils.set_random_seed(812)  # For Keras layers.

In [2]:
# Hugging Face authentication.

from huggingface_hub import login
login()
from google.colab import userdata

# Retrieve the token from Colab secrets
hf_token = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(token=hf_token)

print("Logged in to Hugging Face successfully!")

Logged in to Hugging Face successfully!


In [3]:
import os, random, json, math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print(torch.cuda.get_device_name(0))


Device: cuda
Tesla T4


In [5]:
df = pd.read_excel("/content/Ubuntu Synthetic Dataset.xlsx")
print("Shape:", df.shape)
print(df["label_en"].value_counts())
display(df.head())

Shape: (4000, 16)
label_en
safe           2000
hate_speech    2000
Name: count, dtype: int64


,id,text_en,text_am,label_en,label_am,ubuntu_harm_type_en,ubuntu_harm_type_am,relational_harm_en,relational_harm_am,relation_target_en,relational_target_am,severity_en,severity_am,language,rationale_en,rationale_am
0,1,We should listen before judging.,ከመፍረድ በፊት ማዳመጥ አለብን።,safe,ጥሩ ንግግር,none,የለም,No,የለም,none,የለም,zero,ዜሮ,English-Amharic,"Affirms dignity, cooperation, and mutual respect.",ክብርን፣ ትብብርን እና መከባበርን ያበረታታል።
1,2,We can disagree without hate.,ባንስማማም አለመጣላት እንችላለን።,safe,ጥሩ ንግግር,none,የለም,No,የለም,none,የለም,zero,ዜሮ,English-Amharic,"Affirms dignity, cooperation, and mutual respect.",ክብርን፣ ትብብርን እና መከባበርን ያበረታታል።
2,3,People deserve fairness and care.,ሰዎች ፍትሃዊነት እና እንክብካቤ ይገባቸዋል።,safe,ጥሩ ንግግር,none,የለም,No,የለም,none,የለም,zero,ዜሮ,English-Amharic,"Affirms dignity, cooperation, and mutual respect.",ክብርን፣ ትብብርን እና መከባበርን ያበረታታል።
3,4,Different views can still be handled kindly.,የተለያዩ አስተያየቶች በበጎ ሊታዩ ይችላሉ።,safe,ጥሩ ንግግር,none,የለም,No,የለም,none,የለም,zero,ዜሮ,English-Amharic,"Affirms dignity, cooperation, and mutual respect.",ክብርን፣ ትብብርን እና መከባበርን ያበረታታል።
4,5,Let us find a solution that works for everyone.,ለሁሉም የሚሰራ መፍትሄ እንፈልግ።,safe,ጥሩ ንግግር,none,የለም,No,የለም,none,የለም,zero,ዜሮ,English-Amharic,"Affirms dignity, cooperation, and mutual respect.",ክብርን፣ ትብብርን እና መከባበርን ያበረታታል።


In [6]:
# Build bilingual probing examples.
# Each original row becomes two examples: English and Amharic.
# group_id keeps the translations of the same underlying sentence together
# during splitting, preventing English/Amharic leakage.

examples = []
for _, r in df.iterrows():
    label = 1 if r["label_en"] == "hate_speech" else 0
    group = int(r["id"])
    examples.append({"text": str(r["text_en"]), "language": "en", "label": label, "group": group})
    examples.append({"text": str(r["text_am"]), "language": "am", "label": label, "group": group})

ex = pd.DataFrame(examples)
print("Bilingual examples:", ex.shape)
print(ex["label"].value_counts())
print("Unique underlying groups:", ex["group"].nunique())


Bilingual examples: (8000, 4)
label
0    4000
1    4000
Name: count, dtype: int64
Unique underlying groups: 4000


In [7]:
# Grouped train/validation/test split.
# Because the supplied dataset repeats a small set of templates many times,
# a random row split would produce artificially high scores.
#
# 70% groups -> train, 15% -> validation, 15% -> test.

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(gss1.split(ex, ex["label"], groups=ex["group"]))
train = ex.iloc[train_idx].reset_index(drop=True)
temp = ex.iloc[temp_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_rel, test_rel = next(gss2.split(temp, temp["label"], groups=temp["group"]))
val = temp.iloc[val_rel].reset_index(drop=True)
test = temp.iloc[test_rel].reset_index(drop=True)

print("train:", train.shape, "groups:", train.group.nunique())
print("val:  ", val.shape,   "groups:", val.group.nunique())
print("test: ", test.shape,  "groups:", test.group.nunique())


train: (5600, 4) groups: 2800
val:   (1200, 4) groups: 600
test:  (1200, 4) groups: 600


In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
    output_hidden_states=True
)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded:", MODEL_ID)
print("Hidden size:", model.config.hidden_size)
print("Number of hidden-state layers:", model.config.num_hidden_layers + 1,
      "(embedding output + transformer layers)")


config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Loaded: google/gemma-2-2b
Hidden size: 2304
Number of hidden-state layers: 27 (embedding output + transformer layers)


In [9]:
# Mean-pool token representations.
# We do NOT use the classifier head/logits. The probe receives internal
# representations from Gemma itself.

@torch.no_grad()
def extract_layer_embeddings(texts, layer_idx, batch_size=8, max_length=256):
    outputs = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start+batch_size]
        tok = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        )
        tok = {k: v.to(model.device) for k, v in tok.items()}
        hs = model(**tok, output_hidden_states=True).hidden_states[layer_idx]

        mask = tok["attention_mask"].unsqueeze(-1).to(hs.dtype)
        pooled = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        outputs.append(pooled.float().cpu().numpy())

    return np.concatenate(outputs, axis=0)


In [10]:
# Extract all layers once, saving each layer to disk as float16.
# This avoids keeping hundreds of MB of embeddings in RAM.
#
# On a GPU, batch_size=8 is a conservative starting point for Gemma 2 2B.
# If you run out of VRAM, reduce it to 4 or 2.

EMB_DIR = Path("gemma_layer_embeddings")
EMB_DIR.mkdir(exist_ok=True)

all_ex = pd.concat([train, val, test], ignore_index=True)
texts = all_ex["text"].tolist()

n_layers = model.config.num_hidden_layers + 1

for layer in range(n_layers):
    path = EMB_DIR / f"layer_{layer:02d}.npy"
    if path.exists():
        print("Already exists:", path)
        continue
    emb = extract_layer_embeddings(texts, layer_idx=layer, batch_size=8, max_length=256)
    np.save(path, emb.astype(np.float16))
    print(f"Layer {layer:02d}: {emb.shape} -> {path}")


Layer 00: (8000, 2304) -> gemma_layer_embeddings/layer_00.npy
Layer 01: (8000, 2304) -> gemma_layer_embeddings/layer_01.npy
Layer 02: (8000, 2304) -> gemma_layer_embeddings/layer_02.npy
Layer 03: (8000, 2304) -> gemma_layer_embeddings/layer_03.npy
Layer 04: (8000, 2304) -> gemma_layer_embeddings/layer_04.npy
Layer 05: (8000, 2304) -> gemma_layer_embeddings/layer_05.npy
Layer 06: (8000, 2304) -> gemma_layer_embeddings/layer_06.npy
Layer 07: (8000, 2304) -> gemma_layer_embeddings/layer_07.npy
Layer 08: (8000, 2304) -> gemma_layer_embeddings/layer_08.npy
Layer 09: (8000, 2304) -> gemma_layer_embeddings/layer_09.npy
Layer 10: (8000, 2304) -> gemma_layer_embeddings/layer_10.npy
Layer 11: (8000, 2304) -> gemma_layer_embeddings/layer_11.npy
Layer 12: (8000, 2304) -> gemma_layer_embeddings/layer_12.npy
Layer 13: (8000, 2304) -> gemma_layer_embeddings/layer_13.npy
Layer 14: (8000, 2304) -> gemma_layer_embeddings/layer_14.npy
Layer 15: (8000, 2304) -> gemma_layer_embeddings/layer_15.npy
Layer 16

In [72]:
# Map the concatenated embedding rows back to train/val/test.
n_train, n_val, n_test = len(train), len(val), len(test)
train_slice = slice(0, n_train)
val_slice = slice(n_train, n_train + n_val)
test_slice = slice(n_train + n_val, n_train + n_val + n_test)

y_train = train["label"].to_numpy()
y_val = val["label"].to_numpy()
y_test = test["label"].to_numpy()

results = []
best = None

for layer in range(n_layers):
    X = np.load(EMB_DIR / f"layer_{layer:02d}.npy", mmap_mode="r")

    Xtr = np.asarray(X[train_slice], dtype=np.float32)
    Xv  = np.asarray(X[val_slice], dtype=np.float32)

    probe = Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        ))
    ])

    probe.fit(Xtr, y_train)
    pred = probe.predict(Xv)

    p, r, f1, _ = precision_recall_fscore_support(
        y_val, pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_val, pred)

    row = {"layer": layer, "accuracy": acc, "precision": p, "recall": r, "f1": f1}
    results.append(row)

    if best is None or f1 > best["f1"]:
        best = {
            "layer": layer, "f1": f1, "probe": probe
            }

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)
print("BEST LAYER:", best["layer"], "validation F1:", best["f1"])


,layer,accuracy,precision,recall,f1
0,0,1.0,1.0,1.0,1.0
1,1,1.0,1.0,1.0,1.0
2,2,1.0,1.0,1.0,1.0
3,3,1.0,1.0,1.0,1.0
4,4,1.0,1.0,1.0,1.0
5,5,1.0,1.0,1.0,1.0
6,6,1.0,1.0,1.0,1.0
7,7,1.0,1.0,1.0,1.0
8,8,1.0,1.0,1.0,1.0
9,9,1.0,1.0,1.0,1.0


BEST LAYER: 0 validation F1: 1.0


In [73]:
# Retraining the probe.

best_layer = int(best["layer"])
X = np.load(EMB_DIR / f"layer_{best_layer:02d}.npy", mmap_mode="r")

X_trainval = np.concatenate([
    np.asarray(X[train_slice], dtype=np.float32),
    np.asarray(X[val_slice], dtype=np.float32)
])
y_trainval = np.concatenate([y_train, y_val])

final_probe = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    ))
])
final_probe.fit(X_trainval, y_trainval)

Xtest = np.asarray(X[test_slice], dtype=np.float32)
test_pred = final_probe.predict(Xtest)

print(classification_report(
    y_test, test_pred,
    target_names=["safe", "hate_speech"],
    digits=4,
    zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_test, test_pred))


              precision    recall  f1-score   support

        safe     1.0000    1.0000    1.0000       594
 hate_speech     1.0000    1.0000    1.0000       606

    accuracy                         1.0000      1200
   macro avg     1.0000    1.0000    1.0000      1200
weighted avg     1.0000    1.0000    1.0000      1200

Confusion matrix:
[[594   0]
 [  0 606]]


In [74]:
# Checking whether the probe behaves similarly across English and Amharic.

test_eval = test.copy()
test_eval["prediction"] = test_pred

for lang in ["en", "am"]:
    mask = test_eval["language"].eq(lang)
    p, r, f1, _ = precision_recall_fscore_support(
        test_eval.loc[mask, "label"],
        test_eval.loc[mask, "prediction"],
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(
        test_eval.loc[mask, "label"],
        test_eval.loc[mask, "prediction"]
    )
    print(lang, {"accuracy": acc, "precision": p, "recall": r, "f1": f1})


en {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
am {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


In [75]:
# Ubuntu-aligned analysis.
#
# These metadata fields describe the intended relational harm:
# belonging denial, exclusion, dehumanization, silencing, stereotyping, threat.
# They are NOT fed into the probe.
#
# We use them only after prediction to inspect which kinds of relational harm
# are easier/harder for the latent probe to detect.

test_eval = test_eval.merge(
    df[["id", "ubuntu_harm_type_en", "severity_en"]],
    left_on="group", right_on="id", how="left"
)

print(test_eval.groupby("ubuntu_harm_type_en").apply(
    lambda g: pd.Series({
        "n": len(g),
        "accuracy": accuracy_score(g["label"], g["prediction"])
    }),
    include_groups=False
))


                         n  accuracy
ubuntu_harm_type_en                 
belonging denial     118.0       1.0
dehumanization       106.0       1.0
exclusion             96.0       1.0
none                 594.0       1.0
silencing            118.0       1.0
stereotyping          74.0       1.0
threat                94.0       1.0


In [83]:
import json
import joblib
from pathlib import Path
from transformers import AutoConfig

# Saving the final as classifier.py.

joblib.dump(final_probe, "trained_probe.joblib")

print("Saved trained probe successfully.")
# To prevent AttributeError in case 'model' was overwritten
# by loading 'trained_probe.joblib' into a variable named 'model'.
# We need the config from the original AutoModelForCausalLM.
original_model_config = AutoConfig.from_pretrained(MODEL_ID)
hidden_size_value = int(original_model_config.hidden_size)

metadata = {
    "model_id": MODEL_ID,
    "best_layer": best_layer,
    "hidden_size": hidden_size_value, # Use the correctly obtained hidden_size
    "pooling": "attention-mask mean pooling",
    "probe": "StandardScaler + LogisticRegression",
    "classes": {"0": "safe", "1": "hate_speech"},
    "training_data": "supplied 4000-row English-Amharic synthetic Ubuntu-oriented dataset",
    "input_features": "Gemma hidden states only; harm/rationale metadata excluded",
    "split": "grouped by original sentence id"
}

Path("probe_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
results_df.to_csv("layer_probe_results.csv", index=False)

print("Saved:")
print(" - trained_probe.joblib")
print(" - probe_metadata.json")
print(" - layer_probe_results.csv")

Saved trained probe successfully.
Saved:
 - trained_probe.joblib
 - probe_metadata.json
 - layer_probe_results.csv


In [84]:
from pathlib import Path

print(Path("trained_probe.joblib").exists())

True


In [127]:
from pathlib import Path
import numpy as np
import joblib


class Classifier:
    def __init__(self):
        model_path = Path(__file__).resolve().parent / "trained_probe.joblib"
        self.model = joblib.load(model_path)

    def predict(self, X):
        X = np.asarray(X)
        return self.model.predict(X).astype(int)

In [128]:
!ls

 classifier.py		   probe_metadata.json	 trained_probe.joblib
 gemma_layer_embeddings    sample_data		'Ubuntu Synthetic Dataset.xlsx'
 layer_probe_results.csv   submission.zip


In [132]:
from pathlib import Path
import zipfile

Path("classifier.py").write_text("""
from pathlib import Path
import numpy as np
import joblib


class Classifier:
    def __init__(self):
        model_path = Path(__file__).resolve().parent / "trained_probe.joblib"
        self.model = joblib.load(model_path)

    def predict(self, X):
        X = np.asarray(X)
        return self.model.predict(X).astype(int)
""", encoding="utf-8")

with zipfile.ZipFile("submission_final.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("classifier.py")
    z.write("trained_probe.joblib")

print("ZIP contents:")
with zipfile.ZipFile("submission_final.zip", "r") as z:
    print(z.namelist())
    print("\n--- classifier.py inside ZIP ---")
    print(z.read("classifier.py").decode("utf-8"))

ZIP contents:
['classifier.py', 'trained_probe.joblib']

--- classifier.py inside ZIP ---

from pathlib import Path
import numpy as np
import joblib


class Classifier:
    def __init__(self):
        model_path = Path(__file__).resolve().parent / "trained_probe.joblib"
        self.model = joblib.load(model_path)

    def predict(self, X):
        X = np.asarray(X)
        return self.model.predict(X).astype(int)



In [133]:
from google.colab import files

files.download("submission_final.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>